# WEEK 2 EXERCISE - Deep Research Agent 

In [ ]:
from agents import Agent, WebSearchTool, ModelSettings, function_tool, Runner, trace, gen_trace_id, input_guardrail, output_guardrail, GuardrailFunctionOutput
from pydantic import BaseModel, Field
import os
from typing import Dict
import sendgrid
from sendgrid.helpers.mail import Email, Mail, Content, To
import asyncio
import gradio as gr
from dotenv import load_dotenv

In [ ]:
load_dotenv(override=True)

In [ ]:
HOW_MANY_SEARCHES = 3
MODEL = "gpt-4o-mini"

In [ ]:
search_instructions = (
    "You are a research assistant. Given a search term, you search the web for that term and "
    "produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 "
    "words. Capture the main points. Write succintly, no need to have complete sentences or good "
    "grammar. This will be consumed by someone synthesizing a report, so its vital you capture the "
    "essence and ignore any fluff. Do not include any additional commentary other than the summary itself."
)

search_agent = Agent(
    name="Search agent",
    instructions=search_instructions,
    tools=[WebSearchTool(search_context_size="low")],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)

In [ ]:
# planner agent
planner_instructions = f"You are a helpful research assistant. Given a query, come up with a set of web searches \
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for."


class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")


planner_agent = Agent(
    name="PlannerAgent",
    instructions=planner_instructions,
    model=MODEL,
    output_type=WebSearchPlan,
)

In [ ]:
# writer agent
writer_instructions = (
    "You are a senior researcher tasked with writing a cohesive report for a research query. "
    "You will be provided with the original query, and some initial research done by a research assistant.\n"
    "You should first come up with an outline for the report that describes the structure and "
    "flow of the report. Then, generate the report and return that as your final output.\n"
    "The final output should be in markdown format, and it should be lengthy and detailed. Aim "
    "for 5-10 pages of content, at least 1000 words."
)


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(
    name="WriterAgent",
    instructions=writer_instructions,
    model=MODEL,
    output_type=ReportData,
)

In [ ]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("olasogbayimika@gmail.com")  # Change to your verified sender
    to_email = To("adebimpefolayemi1@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}



In [ ]:
subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [ ]:
email_tools = [subject_tool, html_tool, send_html_email]

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


email_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=email_tools,
    model=MODEL
)

In [ ]:
# input guardrail
class TopicCheckOutput(BaseModel):
    is_inappropriate_topic: bool
    reason: str = Field(description="Explanation of why the topic is inappropriate, if applicable")
    detected_topic: str = Field(description="The problematic topic detected, if any")


guardrail_agent = Agent(
    name="Topic Safety Check",
    instructions="""Check if the research query contains inappropriate or harmful topics that should not be researched.
    
Flag the query if it requests research on:
- Illegal activities or how to commit crimes
- Creating weapons, explosives, or dangerous substances
- Personal private information about specific individuals (doxxing)
- Misinformation or propaganda creation
- Harmful medical advice that could endanger lives

Allow legitimate academic, educational, and professional research topics.""",
    output_type=TopicCheckOutput,
    model=MODEL
)


@input_guardrail
async def guardrail_against_harmful_topics(ctx, agent, message):
    result = await Runner.run(guardrail_agent, message, context=ctx.context)
    is_inappropriate = result.final_output.is_inappropriate_topic
    return GuardrailFunctionOutput(
        output_info={
            "topic_check": result.final_output,
            "reason": result.final_output.reason if is_inappropriate else None
        },
        tripwire_triggered=is_inappropriate
    )

In [ ]:
# output guardrail
class OutputCheckOutput(BaseModel):
    is_problematic_output: bool
    issues_found: list[str] = Field(description="List of issues found in the output, if any")
    severity: str = Field(description="Severity level: 'low', 'medium', or 'high'")


output_guardrail_agent = Agent(
    name="Output Quality Check",
    instructions="""Review the research report output for quality and safety issues.

Flag the output if it contains:
- Fabricated or hallucinated information not supported by research
- Plagiarized content or improper citations
- Biased or one-sided analysis without acknowledging alternative viewpoints
- Sensitive personal information that shouldn't be shared
- Factual inaccuracies or misleading conclusions
- Inappropriate or offensive language

Allow well-researched, balanced, and properly sourced reports.""",
    output_type=OutputCheckOutput,
    model=MODEL
)


@output_guardrail
async def guardrail_check_output_quality(ctx, agent, output):
    result = await Runner.run(output_guardrail_agent, str(output), context=ctx.context)
    is_problematic = result.final_output.is_problematic_output
    return GuardrailFunctionOutput(
        output_info={
            "output_check": result.final_output,
            "issues": result.final_output.issues_found if is_problematic else [],
            "severity": result.final_output.severity if is_problematic else None
        },
        tripwire_triggered=is_problematic and result.final_output.severity == "high"
    )

In [ ]:
class ResearchManager:
    async def run(self, query: str):
        """Run the deep research process, yielding status updates and the final report"""
        trace_id = gen_trace_id()
        with trace("Research trace", trace_id=trace_id):
            print(f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}")
            yield f"View trace: https://platform.openai.com/traces/trace?trace_id={trace_id}"

            # Input guardrail: check for harmful topics
            print("Checking input guardrail...")
            guardrail_result = await Runner.run(guardrail_agent, query)
            if guardrail_result.final_output.is_inappropriate_topic:
                reason = guardrail_result.final_output.reason
                yield f"Sorry, your query was flagged as inappropriate: {reason}"
                return

            print("Starting research...")
            search_plan = await self.plan_searches(query)
            yield "Searches planned, starting to search..."
            search_results = await self.perform_searches(search_plan)
            yield "Searches complete, writing report..."
            report = await self.write_report(query, search_results)

            # Output guardrail: check report quality
            print("Checking output guardrail...")
            output_check = await Runner.run(output_guardrail_agent, report.markdown_report)
            if output_check.final_output.is_problematic_output and output_check.final_output.severity == "high":
                issues = ", ".join(output_check.final_output.issues_found)
                yield f"Report flagged for quality issues: {issues}. Please try a different query."
                return

            yield "Report written, sending email..."
            await self.send_email(report)
            yield "Email sent, research complete"
            yield report.markdown_report

    async def plan_searches(self, query: str) -> WebSearchPlan:
        """Plan the searches to perform for the query"""
        print("Planning searches...")
        result = await Runner.run(planner_agent, f"Query: {query}")
        print(f"Will perform {len(result.final_output.searches)} searches")
        return result.final_output

    async def perform_searches(self, search_plan: WebSearchPlan):
        """Execute all planned searches in parallel"""
        print("Searching...")
        tasks = [asyncio.create_task(self.search(item)) for item in search_plan.searches]
        results = await asyncio.gather(*tasks)
        print("Finished searching")
        return results

    async def search(self, item: WebSearchItem):
        """Run a single web search"""
        input = f"Search term: {item.query}\nReason for searching: {item.reason}"
        result = await Runner.run(search_agent, input)
        return result.final_output

    async def write_report(self, query: str, search_results: list[str]):
        """Write a report based on the search results"""
        print("Thinking about report...")
        input = f"Original query: {query}\nSummarized search results: {search_results}"
        result = await Runner.run(writer_agent, input)
        print("Finished writing report")
        return result.final_output

    async def send_email(self, report: ReportData):
        """Send the report via email"""
        print("Writing email...")
        result = await Runner.run(email_agent, report.markdown_report)
        print("Email sent")
        return report

In [ ]:
# Gradio section
async def run(message: str, history: list):
    async for chunk in ResearchManager().run(message):
        yield chunk

app = gr.ChatInterface(
    run,
    title="Deep Research Agent",
    description="Enter a research topic and the agent will plan searches, gather information, write a detailed report, and email it to you.",
    type="messages",
    examples=[
        "Research the latest advancements in renewable energy technologies",
        "Analyze the current state of artificial intelligence in healthcare",
        "Explore the impact of remote work on productivity and work-life balance"
    ]
)

app.launch()